<a href="https://colab.research.google.com/github/enristra/product-recognition/blob/main/Product_Classification_with_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product Classification con CNN
## Introduzione
Il progetto prende in esame la classificazione tramite CNN del dataset **Grocery Store Dataset** di 81 classi di immagini raffiguranti prodotti alimentari presenti in un supermercato.
Tale progetto è suddiviso in due parti distinte:
- **Task 1**: prevede la progettazione da zero di una rete neurale. In particolare, abbiamo sviluppato 5 diversi modelli di CNN, con l'obiettivo di superare le performance del modello precedente fino al raggiungimento di circa il 60% di validation accuracy;
- **Task 2**: prevede il fine-tuning di una ResNet-18 pre-addestrata su ImageNet sullo stesso dataset. In particolare, il fine-tuning viene effettuato prima con gli stessi iperparametri del miglior modello del Task 1, poi con iperparametri ottimizzati per raggiungere tra l'80% e il 90% di validation accuracy.

In entrambi i task la valutazione finale viene effettuata sul *test set*, riservato esclusivamente alla misurazione delle prestazioni dei modelli migliori.

## Setup dell'ambiente
Eseguiamo adesso il download del dataset da github ed installiamo i pacchetti necessari delle varie librerie che utilizzeremo all'interno del progetto.


In [ ]:
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
!pip install numpy matplotlib pandas tensorflow keras-hub

fatal: destination path 'GroceryStoreDataset' already exists and is not an empty directory.


In [ ]:
from keras import Sequential, Model
from keras.layers import Input, Conv2D,MaxPooling2D, BatchNormalization, Activation,Flatten, Dense, Dropout,GlobalAveragePooling2D, RandomFlip,RandomRotation,RandomZoom,RandomContrast, RandomTranslation, Concatenate
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.utils import set_random_seed
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import pandas as pd
import os
# from google.colab import drive

os.environ["KERAS_BACKEND"] = "tensorflow"

# Task 1
L'obiettivo di questo task è costruire una rete convoluzionale interamente a partire dai layer Keras di base, senza ricorrere ad architetture predefinite. L'approccio adottato è bottom-up: si parte dal modello più semplice possibile (*Model 1*) e si aggiunge complessità passo dopo passo, motivando ogni scelta con l'analisi dei grafici di training precedenti e verificando il miglioramento delle prestazioni.

## Configurazione
La fase di configurazione prevede di impostare le costanti che ci serviranno per l'intero sviluppo del Task 1, compresi gli iperparametri comuni per i modelli di CNN. Ogni scelta è commentata con la sua motivazione.

In [ ]:
DATASET_DIR= '/content/GroceryStoreDataset/dataset/'

# Dimensione delle immagini: abbiamo scelto di utilizzare una dimensione delle immagini diversa da quella originale (192×192 invece che 384×384)
# in modo tale che la RAM non venga saturata durante l'addestramento e che
# non ci sia la perdita di troppi dettagli causati dalla dimensione troppo piccola delle immagini.
IMG_HEIGHT = 192
IMG_WIDTH = 192
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS = 3 # immagini RGB

BATCH_SIZE = 32 # equilibrio tra stabilità del gradiente e utilizzo della RAM GPU.
EPOCHS = 200 # max epochs 200: usiamo EarlyStopping per interrompere prima se necessario.
LEARNING_RATE = 0.001 # valore iniziale per Adam, verrà ridotto automaticamente con ReduceLROnPlateau

NUM_CLASSES  = 81   # classi fine-grained
NUM_SUPER    = 43   # classi coarse/gerarchiche usate solo per il Model 5

DROPOUT_RATE = 0.6 # alto per combattere l'overfitting su un dataset piccolo

# Impostazione del seed per la riproducibilita dei pesi negli esperimenti
SEED = 42
np.random.seed(SEED)
set_random_seed(SEED)


Definiamo anche una funzione che ci permetterà di visualizzare le immagini del dataset.

In [ ]:
def plot(history):
  plt.figure(figsize=(20, 3))
  for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1)
    plt.plot(history.history[metric])
    plt.plot(history.history["val_" + metric])
    plt.title("Model {}".format(metric))
    plt.xlabel("epochs")
    plt.ylabel(metric)
    plt.legend(["train", "val"])

## Eplorazione del dataset
Il Grocery Store Dataset contiene 5125 immagini naturali di prodotti alimentari scattate con la camera degli smartphone in diversi supermercati. Le immagini sono suddivise in:
- **81 classi fine-grained** (es. *Granny Smith Apple*, *Banana*, *Oat Milk* …)
- **43 classi coarse** (macro-categorie, es. *Apple*, *Banana* …)

Il dataset è già suddiviso in train, val e test tramite file `.txt`.

Carichiamo adesso i file `.csv` che contengono le label delle immagini. I file sono strutturati nel modo seguente: Image Path, Class, Coarse Class.

In [ ]:
train_path = os.path.join(DATASET_DIR, 'train.txt')
val_path = os.path.join(DATASET_DIR,'val.txt')
test_path = os.path.join(DATASET_DIR,'test.txt')
classes_path = os.path.join(DATASET_DIR, 'classes.csv')

names = ["Image path", "Class", "Coarse class"]
df_train = pd.read_csv(train_path, names=names)
df_val = pd.read_csv(val_path, names=names)
df_test = pd.read_csv(test_path, names=names)
df_classes = pd.read_csv(classes_path)

df_train.head()

,Image path,Class,Coarse class
0,train/Fruit/Apple/Golden-Delicious/Golden-Deli...,0,0
1,train/Fruit/Apple/Golden-Delicious/Golden-Deli...,0,0
2,train/Fruit/Apple/Golden-Delicious/Golden-Deli...,0,0
3,train/Fruit/Apple/Golden-Delicious/Golden-Deli...,0,0
4,train/Fruit/Apple/Golden-Delicious/Golden-Deli...,0,0


Come possiamo vedere ogni immagine ha due label, il primo specifica la classe più generica, l'altro una classe più specifica. Terremo a mente questa particolare struttura, perché la sfrutteremo nell'ultimo modello.

Mostriamo, adesso, le label generiche e specifiche del dataset.

In [ ]:

unique_classes=df_classes.iloc[:, 0].unique()
unique_coarse_classes=df_classes.iloc[:, 2].unique()

print(f"Number of coarse classes: {len(unique_coarse_classes)}")
print(f"{unique_coarse_classes}\n\n")

print(f"Number of specific classes: {len(unique_classes)}")
print(f"{unique_classes}")

Number of coarse classes: 43
['Apple' 'Avocado' 'Banana' 'Kiwi' 'Lemon' 'Lime' 'Mango' 'Melon'
 'Nectarine' 'Orange' 'Papaya' 'Passion-Fruit' 'Peach' 'Pear' 'Pineapple'
 'Plum' 'Pomegranate' 'Red-Grapefruit' 'Satsumas' 'Juice' 'Milk'
 'Oatghurt' 'Oat-Milk' 'Sour-Cream' 'Sour-Milk' 'Soyghurt' 'Soy-Milk'
 'Yoghurt' 'Asparagus' 'Aubergine' 'Cabbage' 'Carrots' 'Cucumber' 'Garlic'
 'Ginger' 'Leek' 'Mushroom' 'Onion' 'Pepper' 'Potato' 'Red-Beet' 'Tomato'
 'Zucchini']


Number of specific classes: 81
['Golden-Delicious' 'Granny-Smith' 'Pink-Lady' 'Red-Delicious'
 'Royal-Gala' 'Avocado' 'Banana' 'Kiwi' 'Lemon' 'Lime' 'Mango'
 'Cantaloupe' 'Galia-Melon' 'Honeydew-Melon' 'Watermelon' 'Nectarine'
 'Orange' 'Papaya' 'Passion-Fruit' 'Peach' 'Anjou' 'Conference' 'Kaiser'
 'Pineapple' 'Plum' 'Pomegranate' 'Red-Grapefruit' 'Satsumas'
 'Bravo-Apple-Juice' 'Bravo-Orange-Juice' 'God-Morgon-Apple-Juice'
 'God-Morgon-Orange-Juice' 'God-Morgon-Orange-Red-Grapefruit-Juice'
 'God-Morgon-Red-Grapefruit-Juice' 

## Creazione dei dataset
Creiamo adesso una funzione che caricherà le immagini dal percorso del csv, le decodificherà, eseguirà il resize alla dimensione che abbiamo specificato sopra e normalizzerà i valori da interi nel range [0,255] a floating point in [0, 1], poiché le reti neurali lavorano meglio con numeri in virgola mobile.

In [ ]:
def load_image(image_path, label):
    file = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(file, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255
    return image, label

Creiamo, inoltre, una seconda funzione che ci permetterà di convertire i dataframe di pandas in tensori, attraverso i quali costruiremo i nostri dataset su cui lavoreranno i nostri modelli.

In [ ]:
def get_dataset(df, train, super_label=False):
    # Inserimento percorso assoluto delle immagini
    df[names[0]] = df[names[0]].apply(
        lambda i: str(os.path.join(DATASET_DIR, i.strip()))
    )

    image = df[names[0]].values
    # Creazione tensore delle immagini
    x = tf.convert_to_tensor(image, dtype=tf.string)
    # Creazione tensore delle label specifiche
    y_fine = tf.convert_to_tensor(df[names[1]].astype(int).values, dtype=tf.int32)

    if super_label:
        # Creazione tensore delle label grezze per il Model 5 (fine e coarse label)
        y_coarse = tf.convert_to_tensor(df[names[2]].astype(int).values, dtype=tf.int32)
        dataset = tf.data.Dataset.from_tensor_slices((x, {'s': y_coarse, 'f': y_fine}))
    else:
      # Creazione del dataset
        dataset = tf.data.Dataset.from_tensor_slices((x, y_fine))

    # Caricamento immagini
    dataset = dataset.map(load_image)
    if train: dataset = dataset.shuffle(len(df))
    dataset = dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_dataset = get_dataset(df_train, True)
val_dataset = get_dataset(df_val, False)
test_dataset = get_dataset(df_test, False)

Abbiamo così ottenuto tre dataset distinti su cui svolgeremo il training, la validation e il test.

## Input layer e callbacks
Definiamo una funzione che ci permetterà di creare l'input layer, che servirà a definire che tipo di dati dare in ingresso alla rete neurale. Nei modelli faremo uso della data augmentation, che ci aiuterà a mitigare, in parte, i problemi di apprendimento che le reti neurali hanno con dataset troppo piccoli. Le trasformazioni scelte simulano le variabilità realistiche delle fotografie:
- `RandomFlip('horizontal')`: un prodotto può essere fotografato da sinistra o da destra
- `RandomRotation(0.2)`: leggere inclinazioni (±36°) sono comuni in foto scattate a mano
- `RandomZoom(0.2)`: distanze diverse dalla fotocamera producono scale diverse
- `RandomContrast(0.2)`: l'illuminazione dei supermercati non è sempre uniforme
- `RandomTranslation(0.1, 0.1)`: il soggetto non è sempre centrato

Non applichiamo `RandomFlip('vertical')` perchè un prodotto capovolto non è una variazione realistica.

In [ ]:
def input_layer(data_augmentation = True):

    #Costruisce il blocco di input per ogni modello.
    inputs = Input(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS))
    x = inputs   # variabile separata per costruire il grafo

    if data_augmentation:
        x = RandomFlip("horizontal")(x)
        x = RandomRotation(0.2)(x)
        x = RandomZoom(0.2)(x)
        x = RandomContrast(0.2)(x)
        x = RandomTranslation(0.1, 0.1)(x)

    return inputs, x

Per velocizzare e aiutare il modello a convergere usiamo la callback `ReduceLROnPlateau`: se la validation loss non migliora dopo $x$ epoche, vuol dire che stiamo rimbalzando da un fronte all'altro della funzione di loss, quindi il *learning_rate* viene dimezzato in modo automatico, per riprendere la discesa verso il minimo locale.

Allo stesso modo se il modello ha smesso di imparare dopo $y$ epoche vuol dire che il modello ha già raggiunto il minimo locale e ulteriori epoche sono completamente inutili, sfruttiamo così la callback `EarlyStopping` per fermare l'addestramento in anticipo.

L'utlizzo delle callbacks aumenta, tuttavia, il numero di iperparametri da impostare.

In [ ]:
def fit_model(model, monitor_early_stopping="val_accuracy", monitor_reducelr="val_loss", patience_early_stopping=10, patience_reducelr=8):

    #Restituisce i callback standard usati in tutti gli esperimenti dei modelli

    fit = model.fit(train_dataset, validation_data=val_dataset,epochs=EPOCHS, callbacks=[
      EarlyStopping(
          monitor=monitor_early_stopping,
          patience=patience_early_stopping,
          mode="max",
          restore_best_weights=True, # ripristiniamo i pesi migliori
          verbose=1
      ),
      ReduceLROnPlateau(
          monitor=monitor_reducelr,
          factor=0.5,
          patience=patience_reducelr,
          mode="min",
          verbose=1,
          min_lr=1e-5
      ),
    ])
    return fit

## Modello 1 - Baseline CNN
Il primo modello che abbiamo creato presenta 3 blocchi convoluzionali con filtri seguendo il pattern VGG (32 -> 64 -> 128) seguiti da un layer Fully Connected (`Flatten+Dense`) e un output softmax a 81 neuroni. L'ottimizzatore è Adam con impostazioni di default.
Questo modello viene addestrato volutamente senza nessuna data augmentation e nessuna batch normalization: vogliamo vedere le prestazioni del dataset di base senza aiuti per avere una baseline da cui misurare i miglioramenti successivi.

In [ ]:
def model1():
  inputs, x = input_layer(False) # no data augmentation

  # Blocco 1: estrae feature di basso livello (bordi, texture)
  x = Conv2D(filters=32, kernel_size=3, padding="same", activation="relu")(x)
  x = MaxPooling2D()(x)

  # Blocco 2: feature di livello medio (forme, pattern)
  x = Conv2D(filters=64, kernel_size=3, padding="same", activation="relu")(x)
  x = MaxPooling2D()(x)

  # Blocco 3: feature di alto livello (parti di oggetti)
  x = Conv2D(filters=128, kernel_size=3, padding="same", activation="relu")(x)
  x = MaxPooling2D()(x)

  # Flatten: appiattisce le feature map in un vettore 1D
  x = Flatten()(x)

  # Dense FC: combina le feature spaziali in una rappresentazione globale
  x = Dense(units=256, activation="relu")(x)

  # Output: 81 neuroni (una per classe), softmax per probabilità
  outputs = Dense(units=NUM_CLASSES, activation="softmax")(x)

  model = Model(name="model1", inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=Adam(learning_rate=LEARNING_RATE),
      loss='sparse_categorical_crossentropy',
      metrics=["accuracy"]
    )
  return model

In [ ]:
# Costruzione e summary del modello
model1 = model1()
model1.summary()

Model: "model1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 192, 192, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 96, 96, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 48, 48, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73728)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    18,874,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 81)             │        20,817 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,988,689 (72.44 MB)

 Trainable params: 18,988,689 (72.44 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training Model 1
result = fit_model(model1)
plot(result)

# Valutazione Model 1
test = model1.evaluate(test_dataset)
test_loss, test_acc = test[0], test[1]

print(f"Test Accuracy: {test_acc * 100:.2f}%\nTest Loss: {test_loss}")

Epoch 1/200


KeyboardInterrupt: 

### Risultati del modello 1
Dalle metriche del modello e dal grafico di training possiamo vedere subito come il primo modello vada in overfitting rispetto al set di validazione. Ciò è principalmente dovuto alla carenza di esempi (appena 45 per classe), più che alla complessità del modello stesso. Quindi, introduciamo data augmentation per incrementare la dimensione dei dati e mitigare questo problema. Inoltre, la training accuracy cresce a picco mentre la validation accuracy si stabilizza a valori inferiori perciò è stato necessario introdurre anche tecniche di regolarizzazione.

## Modello 2 - Batch Normalization + Data Augmentation
Dato un overfitting pronunciato e training instabile senza normalizzazione del *Model 1*, introduciamo nel *Model 2* le seguenti modifiche:
1. **Batch Normalization dopo ogni layer Conv2D**: normalizza le attivazioni ad ogni mini-batch, riducendo il *covariate shift* interno. Va applicata prima dell'attivazione (schema Conv -> BN -> ReLU)

2. **Data Augmentation**: aumenta artificialmente la diversità del training set applicando trasformazioni casuali alle immagini, costringendo il modello a imparare feature invarianti alle trasformazioni in modo da aumentare la generalizzazione.

3. **Dropout(0.6)** nel classificatore: disattiva casualmente il 60% dei neuroni durante il training, forzando ridondanza nelle rappresentazioni.

4. **Introduzione di un nuovo blocco conv (filtri 16 -> 32 -> 64 -> 128)**: più capacità espressiva grazie alla normalizzazione che ne consente un training più stabile.

In [ ]:
def model2():
    inputs, x = input_layer(True) # data augmentation attiva

    # Blocco 1 — filtri ridotti (16 invece di 32) per compensare i 4 blocchi
    x = Conv2D(filters=16, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    # Blocco 2
    x = Conv2D(filters=32, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    # Blocco 3
    x = Conv2D(filters=64, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    # Blocco 4
    x = Conv2D(filters=128, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Flatten()(x)

    # Abbiamo raddoppiato il numero delle classi da 81 a 162 del Dense e aggiunto un ulteriore layer
    x = Dense(units=162, use_bias=False)(x) # use_bias=False perché la BN ha già il proprio bias
    x = Activation('relu')(x)
    x = BatchNormalization()(x)
    x = Dense(units=162, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(DROPOUT_RATE)(x)

    outputs = Dense(units=81, activation="softmax")(x)

    model = Model(name="model2", inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
model2=model2()
model2.summary()

Model: "model2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 192, 192, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 192, 192, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation              │ (None, 192, 192, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 192, 192, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 192, 192, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 192, 192, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 96, 96, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 96, 96, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 48, 48, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 48, 48, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 3,125,127 (11.92 MB)

 Trainable params: 3,123,999 (11.92 MB)

 Non-trainable params: 1,128 (4.41 KB)

In [ ]:
result = fit_model(model2)
plot(result)

test = model2.evaluate(test_dataset)
test_loss,test_acc=test[0],test[1]

print(f"Test Accuracy: {test_acc * 100:.2f}%\nTest Loss: {test_loss}")

### Risultati del modello 2
Il gap tra training e validation accuracy si riduce notevolmente rispetto al *Model 1*, segno che l'overfitting è stato contenuto. Le curve di loss sono più lisce e la convergenza più regolare grazie alla batch normalization. La validation accuracy mostra un incremento significativo rispetto alla baseline.

## Modello 3 - Global Average Pooling
Il *Model 2* usa `Flatten` dopo gli strati convoluzionali. Con feature map $12 \times 12 \times 128$, il Flatten produce 18.432 elementi quindi inserendo successivamente un Dense si introduce una matrice di pesi $18.432 \times 162 \approx 3$ milioni di parametri. Questo provoca un potenziale overfitting su un dataset piccolo come il nostro.

Perciò, abbiamo pensato di inserire il **Global Average Pooling (GAP)** per risolvere questo problema: invece di appiattire ogni elemento, calcola la media spaziale di ciascun canale, producendo un vettore di dimensione uguale al numero di canali (256 nel quarto blocco). Il Dense successivo avrà $256 \times 128 = 32.768$ parametri, circa due ordini di grandezza in meno. Oltre alla riduzione dei parametri, il GAP ha un effetto regolarizzante intrinseco: aggregando l'informazione con una media, forza il backbone a distribuire le feature discriminative su tutta la feature map anziché in pattern localizzati. Questo rende il modello più robusto alla posizione del soggetto.

Inoltre, abbiamo aumentato la dimensione dei filtri convoluzionali partendo da 32 fino a 256 nell'ultimo blocco ed abbiamo ridotto il Dense ad uno da 128.

In [ ]:
def model3():
    inputs, x = input_layer()

    x = Conv2D(filters=32, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=64, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=128, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=256, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = GlobalAveragePooling2D()(x)

    x = Dense(units=128, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(DROPOUT_RATE)(x)

    outputs = Dense(units=81, activation="softmax")(x)

    model = Model(name="model3", inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
        jit_compile="auto"
    )
    return model


In [ ]:
model3=model3()
model3.summary()

Model: "model3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 192, 192, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 192, 192, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_1            │ (None, 192, 192, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 192, 192, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 192, 192, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 192, 192, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 96, 96, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 96, 96, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 96, 96, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 96, 96, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 48, 48, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 48, 48, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 48, 48, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 24, 24, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 434,065 (1.66 MB)

 Trainable params: 432,849 (1.65 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [ ]:
result = fit_model(model3)
plot(result)

test=model3.evaluate(test_dataset)
test_loss,test_acc=test[0],test[1]

print(f"Test Accuracy: {test_acc * 100:.2f}%\nTest Loss: {test_loss}")

### Risultati del modello 3






## Modello 4

In [ ]:
def model4():
    wd = 1e-4
    inputs, x = input_layer()

    x = Conv2D(filters=32, kernel_size=3, padding="same", kernel_regularizer=l2(wd))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=64, kernel_size=3, padding="same", kernel_regularizer=l2(wd))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=128, kernel_size=3, padding="same", kernel_regularizer=l2(wd))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = Conv2D(filters=256, kernel_size=3, padding="same", kernel_regularizer=l2(wd))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D()(x)

    x = GlobalAveragePooling2D()(x)

    x = Dense(units=128, use_bias=False, kernel_regularizer=l2(wd))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(DROPOUT_RATE)(x)

    outputs = Dense(units=81, activation="softmax")(x)

    model = Model(name="model4", inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
        jit_compile="auto"
    )
    return model


In [ ]:
model4=model4()
model4.summary()

In [ ]:
fit_model(model4)

### Risultati del modello 4

## Modello 5
CNN con multi-task learning per classificazione gerarchica.
Accuracy e generalizzazione migliorano grazie alla condivisione delle rappresentazioni tra i due task

In [ ]:
def model5():
    wd = 1e-3
    inputs, x = input_layer()

    conv = Conv2D(filters=16, kernel_size=3, kernel_regularizer=l2(wd))(x)
    conv = BatchNormalization()(conv)
    conv = Activation('relu')(conv)
    conv = MaxPooling2D()(conv)

    conv = Conv2D(filters=32, kernel_size=3, kernel_regularizer=l2(wd))(conv)
    conv = BatchNormalization()(conv)
    conv = Activation('relu')(conv)
    conv = MaxPooling2D()(conv)

    conv = Conv2D(filters=64, kernel_size=3, kernel_regularizer=l2(wd))(conv)
    conv = BatchNormalization()(conv)
    conv = Activation('relu')(conv)
    conv = MaxPooling2D()(conv)

    conv = Conv2D(filters=128, kernel_size=3, kernel_regularizer=l2(wd))(conv)
    conv = BatchNormalization()(conv)
    conv = Activation('relu')(conv)
    conv = MaxPooling2D()(conv)

    conv = Conv2D(filters=256, kernel_size=3, kernel_regularizer=l2(wd))(conv)
    conv = BatchNormalization()(conv)
    conv = Activation('relu')(conv)
    conv = MaxPooling2D()(conv)



    conv = GlobalAveragePooling2D()(conv)



    dense = Dense(units=128, use_bias=False, kernel_regularizer=l2(wd))(conv)
    dense = BatchNormalization()(dense)
    dense = Activation('relu')(dense)
    dense = Dropout(DROPOUT_RATE)(dense)

    super_class = Dense(units=43, activation='softmax', name='s')(dense)
    combined = Concatenate()([dense, super_class])
    #combined ha 171 valori (feature estratte da cnn + numero di classi coarse grained), questo
    #è ciò che viene passato a fine_class

    fine_class = Dense(units=81, activation='softmax', name='f')(combined)
    model = Model(name = "model5", inputs=inputs, outputs=[super_class, fine_class])
    model.compile(
        optimizer=Adam(LEARNING_RATE),
        jit_compile="auto",
        loss={
            's': 'sparse_categorical_crossentropy',
            'f': 'sparse_categorical_crossentropy'
        },
        loss_weights={
            's': 0.8,
            'f': 1.0
        },
        metrics={
            's': 'accuracy',
            'f': 'accuracy',
        }
    )
    return model

In [ ]:
model5=model5()
model5.summary()

In [ ]:
train_dataset = get_dataset(df_train, True, super_label=True)
val_dataset = get_dataset(df_val, False, super_label=True)
test_dataset = get_dataset(df_test, False,super_label=True)

In [2]:
fit_model(model5, monitor_early_stopping="val_f_accuracy", monitor_reducelr="val_f_loss", patience_early_stopping=40)


NameError: name 'fit_model' is not defined

### Risultati del modello 5

In [1]:
def plot_multitask(history, model_name):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Fine class accuracy
    axes[0, 0].plot(history.history['f_accuracy'], label='Train')
    axes[0, 0].plot(history.history['val_f_accuracy'], label='Val')
    axes[0, 0].set_title("Fine Class (81) - Accuracy")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Fine class loss
    axes[1, 0].plot(history.history['f_loss'], label='Train')
    axes[1, 0].plot(history.history['val_f_loss'], label='Val')
    axes[1, 0].set_title("Fine Class (81) - Loss")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Loss")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Super class accuracy
    axes[0, 1].plot(history.history['s_accuracy'], label='Train')
    axes[0, 1].plot(history.history['val_s_accuracy'], label='Val')
    axes[0, 1].set_title("Super Class (43) - Accuracy")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Super class loss
    axes[1, 1].plot(history.history['s_loss'], label='Train')
    axes[1, 1].plot(history.history['val_s_loss'], label='Val')
    axes[1, 1].set_title("Super Class (43) - Loss")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.suptitle(f"{model_name} - Multi-Task Training", fontsize=14)
    plt.tight_layout()


    plt.show()